# Neutral vs nonneutral: content words only
**CPU; Run all.** Reuses saved feature caches; no EEG reads, re-extraction, or feature preparation.

Keeps only `grammatical_class=content` in BOTH classes. Function words and uncertain/missing grammar are excluded. Sentiment labels remain unchanged: word-alone and word-in-context must agree; sentence sentiment is unrestricted. Neutral is never downsampled. Uses the same logistic/LDA/dummy models and separate sentence/word/reader holdouts.

The grammar labels are existing agent reviews/screens, not human gold POS annotations. The small private TeCo grammar table is already uploaded under `Thesis/gpt uploaded files/word_sentiment_eeg/teco/word_grammar_labels.csv`; its checksum is verified. No private Persian text is in GitHub.

New results: `classification_similarity/neutral_nonneutral_content/`. Estimated extra Drive output **10–40 MB**, with a 60 MB guard. Original results and CNN are unchanged. You can run this later; no need to interrupt an active CNN run.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys, json, html
import pandas as pd
from IPython.display import display, Markdown, Image, HTML

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
ARCHIVE=THESIS/'Results/word_sentiment_eeg/trial_archive_125hz'
SOURCE=THESIS/'Results/word_sentiment_eeg/classification_similarity'
OUT=SOURCE
state=json.loads((ARCHIVE/'manifest.json').read_text())
if state.get('status')!='completed' or state.get('completed_reader_recordings')!=48:
    raise RuntimeError('Preparation is not fully complete. Wait for all 48 recordings; no analysis has started.')
REPO=Path('/content/word-sentiment-eeg-content')
def git(*args):
    subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],check=True,
                   env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary analysis checkout has edits; preserve them before updating.')
    git('-C',REPO,'pull','--ff-only','origin','main')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-analysis.txt')],check=True)
print('Analysis code:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True).strip())
print('Archive complete:',state['completed_reader_recordings'],'recordings')


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/classify_content_neutral_nonneutral.py'),'--thesis-root',str(THESIS)]
run_with_progress(command,OUT/'neutral_nonneutral_content/runtime_status.json',activity='Content words: neutral versus nonneutral using saved features')


## Binary results side by side

Scores are pooled out-of-fold balanced accuracies. Holdouts are separate experiments, not simultaneous exclusion of readers, words and sentences.


In [ ]:
FOLLOW=OUT/'neutral_nonneutral_content'
assert json.loads((FOLLOW/'manifest.json').read_text())['status']=='completed'
old=pd.read_csv(OUT/'scores.csv')
old=old[(old.policy=='word_agreement') & (old.target=='binary') & (old.feature=='post')].copy()
old['comparison']='positive vs negative'
new=pd.read_csv(FOLLOW/'scores.csv')
new=new[new.feature=='post'].copy()
new['comparison']='neutral vs nonneutral'
summary=pd.concat([old,new],ignore_index=True)
summary['balanced_accuracy_percent']=(summary.balanced_accuracy*100).round(2)
display(summary[['dataset','task','comparison','split','model','balanced_accuracy_percent','trials','class_counts']])
display(Markdown('### Pre-onset and word/fixation controls'))
controls=pd.read_csv(FOLLOW/'scores.csv')
display(controls[controls.feature.isin(['pre','nuisance'])][['dataset','task','feature','balanced_accuracy','roc_auc']])
skipped=json.loads((FOLLOW/'skipped.json').read_text())
if skipped: display(skipped)


## ZUCO SR: neutral/nonneutral confusion matrices


In [ ]:
plot=FOLLOW/'zuco'/'SR'/'confusions.png'
if plot.is_file(): display(Image(filename=str(plot)))


## ZUCO NR: neutral/nonneutral confusion matrices


In [ ]:
plot=FOLLOW/'zuco'/'NR'/'confusions.png'
if plot.is_file(): display(Image(filename=str(plot)))


## ZUCO TSR: neutral/nonneutral confusion matrices


In [ ]:
plot=FOLLOW/'zuco'/'TSR'/'confusions.png'
if plot.is_file(): display(Image(filename=str(plot)))


## TECO task1: neutral/nonneutral confusion matrices


In [ ]:
plot=FOLLOW/'teco'/'task1'/'confusions.png'
if plot.is_file(): display(Image(filename=str(plot)))


## What changed after removing function words


In [ ]:
CONTENT=OUT/'neutral_nonneutral_content'
display(pd.read_csv(CONTENT/'selection_counts.csv'))
new=pd.read_csv(CONTENT/'scores.csv')
old=pd.read_csv(OUT/'neutral_nonneutral/scores.csv')
new=new[(new.feature=='post')&(new.model=='logistic')]
old=old[(old.feature=='post')&(old.model=='logistic')]
comparison=new.merge(old[['dataset','task','split','balanced_accuracy']],on=['dataset','task','split'],suffixes=('_content_only','_all_words'))
comparison['content_only_percent']=100*comparison.balanced_accuracy_content_only
comparison['all_words_percent']=100*comparison.balanced_accuracy_all_words
display(comparison[['dataset','task','split','content_only_percent','all_words_percent']].round(2))
print((CONTENT/'manifest.json').read_text())
